In [ ]:
import numpy as np

In [ ]:
import os
from pathlib import Path

In [ ]:
import matplotlib as mplt
from matplotlib import pyplot as plt
import seaborn as sns

In [ ]:
root = Path("/workspace/existing_result_uq_boxplot")

In [ ]:
PALETTES = [
    {"line": "#9A6A00", "band1": "#E69F00", "band2": "#E69F00", "marker": "o"},  # orange
    {"line": "#1A7AAF", "band1": "#56B4E9", "band2": "#56B4E9", "marker": "s"},  # sky blue
    {"line": "#006B4E", "band1": "#009E73", "band2": "#009E73", "marker": "^"},  # green
]

In [ ]:
rc = {
    "font.family":     "serif",
    "font.size":       9,
    "axes.labelsize":  9,
    "axes.titlesize":  9,
    "legend.fontsize": 10,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "figure.figsize":  (6.5, 3.4),
    "figure.constrained_layout.use": True,
    "figure.autolayout":             False,
    "savefig.pad_inches":            0.015,
}

In [ ]:
mplt.rcParams.update(rc)

In [ ]:

_GRID = "#E8E0DF"

def plot_publication_boxplots(files_list, labels_list, filename, keys=['total', 'aleatoric', 'epistemic'], normalise_classification=True):
    """
    files_list: List of paths to .npz files
    labels_list: Names for each panel (e.g., ['Dataset 1', 'Dataset 2'])
    """
    n_panels = len(files_list)
    fig, axes = plt.subplots(1, n_panels, figsize=rc["figure.figsize"], sharey=True)

    # Ensure axes is iterable even if there is only one panel
    if n_panels == 1:
        axes = [axes]

    for i, (npz_path, label) in enumerate(zip(files_list, labels_list)):
        ax = axes[i]
        data_file = np.load(npz_path)

        norm_constant = None
        if normalise_classification and label.lower() in ("cora", "citeseer", "tolokers2"):
            if label.lower() == "cora":
                norm_constant = np.log(7)
            elif label.lower() == "tolokers2":
                norm_constant = np.log(2)
            elif label.lower() == "citeseer":
                norm_constant = np.log(6)
        
        # Extract and flatten (N,1) -> (N,)
        plot_data = []
        for k in keys:
            if normalise_classification and label.lower() in ("cora", "citeseer", "tolokers2"):
                plot_data.append(data_file[k].flatten()/norm_constant)
            else:
                plot_data.append(data_file[k].flatten())

        ratio_val = np.mean(data_file["epistemic"]) / np.mean(data_file["aleatoric"])
        ax.text(0.95, 0.5, fr"$\text{{Epi}} / \text{{Ale}} \approx {ratio_val:.0e}$",
                transform=ax.transAxes, 
                fontsize=12,
                rotation="vertical",
                #verticalalignment='top',
                fontweight="bold",
                horizontalalignment='right',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.5, edgecolor='none'))
        
        # Create boxplot
        # patch_artist=True allows us to fill the boxes with color
        bp = ax.boxplot(plot_data, patch_artist=True, widths=0.6,
                        medianprops=dict(color="white", linewidth=1),
                        flierprops=dict(marker='o', markersize=4, linestyle='none'))
        if "QM9" in labels_list:
            ax.set_ylim(top=1)    
        # Apply your custom palette to the boxes
        for patch, color_set in zip(bp['boxes'], PALETTES):
            patch.set_facecolor(color_set["band1"])
            patch.set_edgecolor(color_set["line"])
            patch.set_linewidth(1.5)

        # Apply line colors to whiskers and caps
        for j in range(len(keys)):
            color = PALETTES[j]["line"]
            bp['whiskers'][2*j].set_color(color)
            bp['whiskers'][2*j+1].set_color(color)
            bp['caps'][2*j].set_color(color)
            bp['caps'][2*j+1].set_color(color)
            # Outlier (fliers) styling
            bp['fliers'][j].set_markerfacecolor(PALETTES[j]["band2"])
            bp['fliers'][j].set_markeredgecolor(color)

        # Styling the panel
        ax.set_title(label, fontsize=14, fontweight='bold', pad=0)
        ax.set_xticklabels(["Tot", "Ale", "Epi"], fontsize=15)
        ax.set_axisbelow(True) # Ensure grid is behind plots
        ax.tick_params(axis='y', labelsize=15)
        ax.grid(True, axis='y', linestyle='--', alpha=0.7, color=_GRID)
        
        # Despine for cleanliness
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('#333333')
        ax.spines['bottom'].set_color('#333333')

    # plt.tight_layout()
    plt.savefig(filename+".pdf", dpi=800, ) # bbox_inches="tight"
    return fig

In [ ]:
_ = plot_publication_boxplots([root/"uq_breakdown_artnetviews"/"uq_breakdown_25x10.npz", 
                               root/"uq_breakdown_chameleon"/"uq_breakdown_25x10.npz", 
                               root/"uq_breakdown_gapsmallqm9"/"uq_breakdown_25x10.npz", ], # root/"uq_breakdown_pems"/"uq_breakdown_25x10.npz"
                              ["Artnetviews", "Chameleon", "QM9"],
                             "regression_uncertainty_breakdown")

In [ ]:
max_epistemic_mean = 0
num_models = -1
for i in [1,2,3,4,5,10,15,25]:
    uqs = np.load(root / "uq_breakdown_pems" / f"uq_breakdown_{i}x10.npz")
    epis_mean_current = uqs["epistemic"].mean()
    if epis_mean_current > max_epistemic_mean:
        max_epistemic_mean = epis_mean_current
        num_models = i
print(i,max_epistemic_mean)

In [ ]:
_ = plot_publication_boxplots([root/"uq_breakdown_citeseer"/"uq_breakdown_25x10.npz", 
                               root/"uq_breakdown_cora"/"uq_breakdown_25x10.npz",
                               root/"uq_breakdown_tolokers2"/"uq_breakdown_25x10.npz"], 
                               ["Citeseer", "Cora", "Tolokers2"],
                             "classificaiton_uncertainty_breakdown_normalised", normalise_classification=True)